<a href="https://colab.research.google.com/github/micah-shull/AI_Agents/blob/main/889_MSOv3_Telemetry.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Telemetry Loader — Mission Orchestrator V3

The `load_telemetry` function is responsible for loading the **operational telemetry datasets** that power the Mission Orchestrator’s intelligence engine.

Rather than relying on AI models to interpret raw activity logs, the system gathers **structured operational data** from well-defined telemetry files. These datasets form the factual foundation from which the orchestrator derives metrics, signals, and executive insights.

This approach ensures the system’s conclusions are **data-driven, reproducible, and auditable**.

---

# Why Telemetry Matters in an AI Operations System

Most AI agent implementations rely heavily on model reasoning or prompt chains to determine system behavior.

That approach can produce impressive demos, but it often lacks the operational transparency required in real environments.

This orchestrator takes a different approach:

1. Collect **structured telemetry** about mission activity.
2. Convert telemetry into **deterministic intelligence metrics**.
3. Generate **decision-ready operational signals**.

The telemetry loader is therefore the **entry point into the system’s operational intelligence pipeline**.

Without this step, later analysis stages would lack the reliable data required to produce trustworthy insights.

---

# Core Responsibility of the Loader

The `load_telemetry` function gathers all datasets required to evaluate mission performance.

These datasets include:

```
mission_runs
task_execution_logs
mission_risk_events
agent_performance_stats
mission_portfolio_snapshot
mission_kpis
```

Each dataset captures a different dimension of system behavior.

For example:

* **mission_runs** records overall mission outcomes
* **task_execution_logs** capture individual task activity
* **mission_risk_events** identify operational threats
* **agent_performance_stats** measure agent reliability
* **mission_portfolio_snapshot** summarizes system health
* **mission_kpis** define mission performance targets

By loading these datasets into the orchestrator state, the system gains a complete operational picture of how missions are performing.

---

# Flexible Data Location Handling

The loader supports flexible project layouts by determining the data directory dynamically.

The logic follows a simple hierarchy:

1. If the state already provides a `data_dir`, use it.
2. Otherwise fall back to the directory defined in the orchestrator configuration.

This allows the orchestrator to run in multiple environments, including:

* local development notebooks
* testing pipelines
* production deployments

Operational systems benefit greatly from this kind of **configurable data access pattern**.

---

# Config-Driven Telemetry Files

Instead of hardcoding filenames, the loader reads them from the configuration object.

Example configuration values include:

```
mission_runs_file
task_execution_logs_file
mission_risk_events_file
agent_performance_stats_file
mission_portfolio_snapshot_file
mission_kpis_file
```

This design improves system flexibility and reduces the risk of fragile code.

If a telemetry file is renamed or replaced, the change can be handled in configuration without modifying the loader logic.

For organizations operating AI systems at scale, this separation of **code and operational configuration** is extremely important.

---

# Mission-Level Filtering

Many orchestrators operate across multiple missions.

To support this, the loader filters mission runs based on the mission identifier supplied in the state.

If a `mission_id` is provided:

```
only runs belonging to that mission are loaded
```

If no mission is specified:

```
all runs are available
```

This allows the same telemetry store to support:

* **single-mission analysis**
* **portfolio-wide analysis**

without needing separate datasets.

---

# Safe Data Loading

The helper function `_load_json` protects the system from missing telemetry files.

If a file does not exist:

```
None is returned
```

The loader then converts this into an empty dataset when appropriate.

This prevents the orchestrator from crashing when telemetry is incomplete or partially populated.

Operational systems must tolerate imperfect data environments, and this design supports that requirement.

---

# Mission KPI Selection

The system supports both **mission-specific KPI definitions** and global KPI structures.

If a `mission_id` is supplied, the loader extracts only the KPIs relevant to that mission.

Example structure:

```
mission_kpis.json
{
    "M001": {...},
    "M002": {...}
}
```

This allows each mission to define its own performance metrics and targets.

By loading the appropriate KPI set into the state, later analysis stages can calculate mission efficiency correctly.

---

# Derived Metadata

The loader also adds two pieces of metadata to the state.

### Run Count

```
state["run_count"]
```

This simplifies later intelligence calculations such as:

* trend detection
* cold start handling
* data quality signals

Instead of recalculating run counts repeatedly, the value is available immediately.

---

### Schema Version

```
state["schema_version"] = "v3-telemetry-1.0"
```

Schema versioning is an important practice in operational data systems.

It ensures that future changes to telemetry formats can be tracked and handled safely.

For example, if new telemetry fields are introduced in V4, the system can detect the version and adjust logic accordingly.

---

# Why This Pattern Builds Trust

This loader reinforces several key architectural principles that make the system trustworthy.

### Deterministic Data Sources

All intelligence calculations rely on explicit telemetry datasets rather than model interpretation.

### Explicit Data Flow

Telemetry is loaded directly into the orchestrator state, making the system’s reasoning traceable.

### Configurable Infrastructure

Operational parameters such as file locations and telemetry sources are controlled through configuration.

### Failure Tolerance

The loader gracefully handles missing telemetry instead of crashing the system.

Together, these patterns create an AI system that behaves more like a **traditional operational platform** than an experimental agent.

---

# Small Improvements Worth Considering

The implementation is already strong. A few minor improvements could make it even more robust.

---

### 1️⃣ Initialize Error Tracking

Earlier we defined an `errors` list in the state. It may be helpful to ensure it always exists.

Example:

```
state.setdefault("errors", [])
```

This allows later nodes to safely append error messages.

---

### 2️⃣ Validate Mission ID

If a mission identifier is supplied but does not exist in the KPI dataset, the system could log a warning.

Example scenario:

```
mission_id = M003
mission_kpis.json only contains M001 and M002
```

Capturing this situation early can prevent confusing downstream results.

---

### 3️⃣ Optional Dataset Logging

For debugging or operational monitoring, it could be useful to record how many records were loaded.

Example metadata:

```
telemetry_summary = {
    "runs_loaded": len(mission_runs_filtered),
    "tasks_loaded": len(task_execution_logs),
    "risk_events_loaded": len(mission_risk_events)
}
```

This can help operators quickly verify telemetry integrity.

---

# Architectural Impact

This telemetry loader represents the **foundation of the orchestrator’s intelligence system**.

Once telemetry is loaded into the state, the next stages of the pipeline will:

```
compute_signature_metrics
→ determine_status_signals
→ generate_executive_report
```

Each stage builds upon the structured telemetry collected here.

By separating telemetry ingestion from intelligence calculations, the architecture remains:

* modular
* testable
* extensible

This separation is a key reason the system can evolve from **descriptive intelligence (V3)** to **predictive intelligence (V4)**.




In [ ]:
from __future__ import annotations

import json
from dataclasses import asdict
from pathlib import Path
from typing import Any, Dict, List, Optional

from config import MissionOrchestratorV3Config, MissionOrchestratorV3State


def _load_json(path: Path) -> Any:
    if not path.exists():
        return None
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def load_telemetry(
    state: MissionOrchestratorV3State,
    config: MissionOrchestratorV3Config,
    project_root: Optional[Path] = None,
) -> MissionOrchestratorV3State:
    """Load core telemetry datasets for MSO v3 into state."""
    root = project_root or Path(__file__).resolve().parents[4]

    data_dir = Path(state.get("data_dir") or config.data_dir)
    data_path = root / data_dir

    cfg = asdict(config)

    mission_runs = _load_json(data_path / cfg["mission_runs_file"]) or []
    task_execution_logs = _load_json(data_path / cfg["task_execution_logs_file"]) or []
    mission_risk_events = _load_json(data_path / cfg["mission_risk_events_file"]) or []
    agent_performance_stats = _load_json(data_path / cfg["agent_performance_stats_file"]) or []
    mission_portfolio_snapshot = _load_json(data_path / cfg["mission_portfolio_snapshot_file"])
    mission_kpis_all = _load_json(data_path / cfg["mission_kpis_file"]) or {}

    mission_id = state.get("mission_id")
    mission_kpis: Dict[str, Any]
    if mission_id and isinstance(mission_kpis_all, dict):
        mission_kpis = mission_kpis_all.get(mission_id, {})
    else:
        mission_kpis = mission_kpis_all

    # Filter runs to this mission if mission_id provided
    mission_runs_filtered: List[Dict[str, Any]] = []
    for run in mission_runs:
        if mission_id is None or run.get("mission_id") == mission_id:
            mission_runs_filtered.append(run)

    state["mission_runs"] = mission_runs_filtered
    state["task_execution_logs"] = task_execution_logs
    state["mission_risk_events"] = mission_risk_events
    state["agent_performance_stats"] = agent_performance_stats
    state["mission_portfolio_snapshot"] = mission_portfolio_snapshot
    state["mission_kpis"] = mission_kpis

    state["run_count"] = len(mission_runs_filtered)
    state.setdefault("schema_version", "v3-telemetry-1.0")

    return state

